# Pipeline 2: 方面情感分析模型

本Notebook完成以下任务：
1. 加载预处理后的数据
2. 微调RoBERTa模型进行4分类情感分析（positive/negative/neutral/conflict）
3. 评估模型性能
4. 保存并上传到Hugging Face

## 3.1 安装和导入必要的库

In [ ]:
# 安装必要的库
!pip install -q transformers datasets accelerate evaluate scikit-learn huggingface_hub

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from torch import nn
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    TrainingArguments, 
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset, DatasetDict
from sklearn.metrics import (
    accuracy_score, 
    f1_score, 
    precision_score, 
    recall_score, 
    classification_report,
    confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
from huggingface_hub import notebook_login
import os

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

# 检查GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")
print(f"✓ 库导入成功")

## 3.2 加载数据

In [ ]:
# 定义情感标签
SENTIMENT_LABELS = ['positive', 'negative', 'neutral', 'conflict']
NUM_LABELS = len(SENTIMENT_LABELS)

# 加载数据
with open('GroupXX_Dataset_files/train_sentiment.json', 'r', encoding='utf-8') as f:
    train_data = json.load(f)

with open('GroupXX_Dataset_files/trial_sentiment.json', 'r', encoding='utf-8') as f:
    val_data = json.load(f)

print(f"训练集: {len(train_data)} 条")
print(f"验证集: {len(val_data)} 条")
print(f"情感标签: {SENTIMENT_LABELS}")

In [ ]:
# 查看数据样本
df_train = pd.DataFrame(train_data)
print("\n训练集数据分布:")
print(df_train['sentiment'].value_counts())

print("\n数据示例:")
print(df_train.head())

## 3.3 构建输入格式

将方面信息融入输入文本，例如：
- 输入: "[ASPECT] food [TEXT] The pasta was delicious."
- 输出: "positive"

In [ ]:
def prepare_input_format(data):
    """
    准备输入格式：将方面类别信息融入文本
    格式: [ASPECT] aspect [TEXT] text
    """
    processed_data = []
    for item in data:
        # 构建组合文本
        combined_text = f"[ASPECT] {item['aspect']} [TEXT] {item['text']}"
        processed_data.append({
            'text': combined_text,
            'label': item['sentiment_id'],
            'original_text': item['text'],
            'aspect': item['aspect'],
            'sentiment': item['sentiment']
        })
    return processed_data

train_processed = prepare_input_format(train_data)
val_processed = prepare_input_format(val_data)

print("输入格式示例:")
print(f"文本: {train_processed[0]['text']}")
print(f"标签: {train_processed[0]['label']} ({train_processed[0]['sentiment']})")

In [ ]:
# 转换为Hugging Face Dataset格式
def create_dataset(processed_data):
    """创建Hugging Face Dataset"""
    texts = [item['text'] for item in processed_data]
    labels = [item['label'] for item in processed_data]
    
    return Dataset.from_dict({
        'text': texts,
        'labels': labels
    })

train_dataset = create_dataset(train_processed)
val_dataset = create_dataset(val_processed)

print(f"\n训练集大小: {len(train_dataset)}")
print(f"验证集大小: {len(val_dataset)}")

## 3.4 加载预训练模型和Tokenizer

In [ ]:
# 选择基础模型（使用RoBERTa-base，通常比BERT在情感分析上表现更好）
MODEL_NAME = "roberta-base"  # 可选: bert-base-uncased, distilbert-base-uncased

# 加载Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 加载模型（单标签分类）
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=NUM_LABELS,
    id2label={i: label for i, label in enumerate(SENTIMENT_LABELS)},
    label2id={label: i for i, label in enumerate(SENTIMENT_LABELS)}
).to(device)

print(f"✓ 模型 {MODEL_NAME} 加载成功")
print(f"  - 类别数: {NUM_LABELS}")
print(f"  - 问题类型: 单标签分类")

## 3.5 数据预处理（Tokenization）

In [ ]:
# 定义tokenize函数
def tokenize_function(examples):
    """对文本进行tokenization"""
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=256
    )

# 应用tokenization
train_tokenized = train_dataset.map(tokenize_function, batched=True)
val_tokenized = val_dataset.map(tokenize_function, batched=True)

# 转换为torch tensor格式
train_tokenized.set_format(
    type="torch", 
    columns=["input_ids", "attention_mask", "labels"]
)
val_tokenized.set_format(
    type="torch", 
    columns=["input_ids", "attention_mask", "labels"]
)

print("✓ Tokenization完成")
print(f"训练集大小: {len(train_tokenized)}")
print(f"验证集大小: {len(val_tokenized)}")

## 3.6 定义评估指标

In [ ]:
def compute_metrics(eval_pred):
    """
    计算分类评估指标
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    
    # 计算指标
    accuracy = accuracy_score(labels, predictions)
    f1_weighted = f1_score(labels, predictions, average='weighted')
    f1_macro = f1_score(labels, predictions, average='macro')
    precision_weighted = precision_score(labels, predictions, average='weighted', zero_division=0)
    recall_weighted = recall_score(labels, predictions, average='weighted', zero_division=0)
    
    return {
        'accuracy': accuracy,
        'f1_weighted': f1_weighted,
        'f1_macro': f1_macro,
        'precision_weighted': precision_weighted,
        'recall_weighted': recall_weighted
    }

print("✓ 评估指标函数定义完成")

## 3.7 设置训练参数

In [ ]:
# 定义训练参数
training_args = TrainingArguments(
    output_dir='./results_pipeline2',
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    logging_dir='./logs_pipeline2',
    logging_steps=50,
    save_total_limit=2,
    push_to_hub=False,  # 训练完成后再手动上传
)

# 创建Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

print("✓ Trainer创建完成")
print(f"  - 学习率: {training_args.learning_rate}")
print(f"  - Batch size: {training_args.per_device_train_batch_size}")
print(f"  - 训练轮数: {training_args.num_train_epochs}")

## 3.8 开始训练

In [ ]:
# 开始训练
print("开始训练模型...")
trainer.train()

## 3.9 模型评估

In [ ]:
# 在验证集上进行评估
print("\n在验证集上评估...")
eval_results = trainer.evaluate()

print("\n===== 验证集评估结果 =====")
for key, value in eval_results.items():
    print(f"{key}: {value:.4f}")

In [ ]:
# 获取详细分类报告
predictions = trainer.predict(val_tokenized)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

print("\n===== 分类报告 =====")
print(classification_report(
    labels, 
    preds, 
    target_names=SENTIMENT_LABELS,
    zero_division=0
))

In [ ]:
# 绘制混淆矩阵
cm = confusion_matrix(labels, preds)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, 
    annot=True, 
    fmt='d', 
    cmap='Blues',
    xticklabels=SENTIMENT_LABELS,
    yticklabels=SENTIMENT_LABELS
)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix - Sentiment Analysis')
plt.tight_layout()
plt.savefig('GroupXX_Dataset_files/confusion_matrix_pipeline2.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ 混淆矩阵已保存")

## 3.10 保存模型

In [ ]:
# 保存模型到本地
MODEL_SAVE_PATH = 'GroupXX_Dataset_files/Fine-tuned_Model_files/sentiment_analysis_model'

# 创建目录
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)

# 保存模型和tokenizer
trainer.save_model(MODEL_SAVE_PATH)
tokenizer.save_pretrained(MODEL_SAVE_PATH)

print(f"✓ 模型已保存到: {MODEL_SAVE_PATH}")

## 3.11 测试模型推理

In [ ]:
# 测试模型推理
def predict_sentiment(text, aspect, model, tokenizer):
    """预测特定方面的情感"""
    # 构建输入格式
    combined_text = f"[ASPECT] {aspect} [TEXT] {text}"
    
    inputs = tokenizer(
        combined_text, 
        return_tensors="pt", 
        truncation=True, 
        padding=True,
        max_length=256
    ).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()[0]
    
    # 获取预测结果
    pred_id = np.argmax(probs)
    sentiment = SENTIMENT_LABELS[pred_id]
    
    return {
        'sentiment': sentiment,
        'confidence': float(probs[pred_id]),
        'probabilities': {
            SENTIMENT_LABELS[i]: float(probs[i]) 
            for i in range(len(SENTIMENT_LABELS))
        }
    }

# 测试示例
test_cases = [
    {
        "text": "The food was absolutely delicious and beautifully presented!",
        "aspect": "food"
    },
    {
        "text": "The waiter was very rude and ignored us for 20 minutes.",
        "aspect": "service"
    },
    {
        "text": "The prices are quite reasonable for the quality.",
        "aspect": "price"
    },
    {
        "text": "The music was a bit loud but the decor was nice.",
        "aspect": "ambience"
    }
]

print("===== 模型推理测试 =====\n")
for test in test_cases:
    result = predict_sentiment(test["text"], test["aspect"], model, tokenizer)
    print(f"文本: {test['text']}")
    print(f"方面: {test['aspect']}")
    print(f"预测情感: {result['sentiment']} (置信度: {result['confidence']:.3f})")
    print("-" * 80)

## 3.12 上传到Hugging Face Hub

In [ ]:
# 登录Hugging Face（需要在Colab中输入token）
# notebook_login()

In [ ]:
# 上传到Hugging Face Hub
# 请替换为你的Hugging Face用户名和模型名称
HUB_MODEL_NAME = "your-username/absa-sentiment-analysis"

# 推送模型到Hub
# trainer.push_to_hub(HUB_MODEL_NAME)
# tokenizer.push_to_hub(HUB_MODEL_NAME)

print(f"模型上传命令（请在本地执行）:")
print(f"  huggingface-cli login")
print(f"  transformers-cli upload {MODEL_SAVE_PATH}")
print(f"\n或使用代码: trainer.push_to_hub('{HUB_MODEL_NAME}')")

## 3.13 记录实验结果

In [ ]:
# 记录实验结果
experiment_results = {
    "task": "Pipeline 2 - 方面情感分析",
    "base_model": MODEL_NAME,
    "num_labels": NUM_LABELS,
    "label_names": SENTIMENT_LABELS,
    "hyperparameters": {
        "learning_rate": training_args.learning_rate,
        "batch_size": training_args.per_device_train_batch_size,
        "num_epochs": training_args.num_train_epochs,
        "weight_decay": training_args.weight_decay,
        "max_length": 256,
        "input_format": "[ASPECT] aspect [TEXT] text"
    },
    "dataset": {
        "train_size": len(train_data),
        "val_size": len(val_data)
    },
    "evaluation_results": {
        "accuracy": eval_results.get('eval_accuracy', 0),
        "f1_weighted": eval_results.get('eval_f1_weighted', 0),
        "f1_macro": eval_results.get('eval_f1_macro', 0),
        "precision_weighted": eval_results.get('eval_precision_weighted', 0),
        "recall_weighted": eval_results.get('eval_recall_weighted', 0)
    },
    "model_path": MODEL_SAVE_PATH,
    "hub_model_name": HUB_MODEL_NAME
}

# 保存实验结果
with open('GroupXX_Dataset_files/experiment_pipeline2.json', 'w', encoding='utf-8') as f:
    json.dump(experiment_results, f, indent=2, ensure_ascii=False)

print("===== Pipeline 2 实验结果 =====")
print(json.dumps(experiment_results, indent=2, ensure_ascii=False))

print("\n✓ Pipeline 2 完成！")